In [2]:
%pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
You should consider upgrading via the '/Users/jaskirat_singh_grewal/Documents/IDC409_Project/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter


from pathlib import Path

# Your notebook is in IDC409_Project/codes/
ROOT = Path.cwd().parent if Path.cwd().name == "codes" else Path.cwd()

# Input CSV files
INPUT_DATA = ROOT / "data" / "Extra_Files"

MATH_FILE = INPUT_DATA / "student-mat.csv"
POR_FILE = INPUT_DATA / "student-por.csv"

# Output Excel file
OUTPUT = ROOT / "data" / "student_data.xlsx"

# Verify the paths
print("Project root:", ROOT)
print("Math CSV exists:", MATH_FILE.exists())
print("Portuguese CSV exists:", POR_FILE.exists())
print("Output path:", OUTPUT)

def load_csv(path):
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Put the original CSV in data/.")
    return pd.read_csv(path, sep=";")

def encode_subject(df, subject):
    # Separate target before encoding so G3 cannot accidentally become a feature.
    y = df["G3"].astype(float).to_numpy()
    X = df.drop(columns=["G3"]).copy()
    categorical = X.select_dtypes(include=["object", "category"]).columns.tolist()
    X = pd.get_dummies(X, columns=categorical, dtype=int, prefix_sep="__")
    X.insert(0, "Subject", subject)
    X["G3_math"] = y if subject == "Mathematics" else np.nan
    X["G3_portuguese"] = y if subject == "Portuguese" else np.nan
    X["G3_target"] = y
    return X

def main():
    math = encode_subject(load_csv(MATH_FILE), "Mathematics")
    portuguese = encode_subject(load_csv(POR_FILE), "Portuguese")
    # Align category columns in case a category occurs in only one dataset.
    math, portuguese = math.align(portuguese, join="outer", axis=1, fill_value=0)
    special = ["Subject", "G3_math", "G3_portuguese", "G3_target"]
    features = sorted(c for c in math.columns if c not in special)
    ordered = special + features
    math, portuguese = math[ordered], portuguese[ordered]
    combined = pd.concat([math, portuguese], ignore_index=True)

    descriptions = [
        ["Subject", "Text", "Mathematics or Portuguese; identifies the subject."],
        ["G3_math", "Target", "Final mathematics grade; populated only for mathematics rows."],
        ["G3_portuguese", "Target", "Final Portuguese grade; populated only for Portuguese rows."],
        ["G3_target", "Target", "Final grade for the subject in Subject."],
        ["school__GP", "0/1 indicator", "1 for GP (Gabriel Pereira), otherwise 0."],
        ["school__MS", "0/1 indicator", "1 for MS (Mousinho da Silveira), otherwise 0."],
        ["Categorical features", "One-hot encoded", "Each category has a separate 0/1 indicator column."],
        ["Numeric features", "Numeric", "Original numeric values retained; fit any scaling on training data only."],
        ["G1 and G2", "Numeric features", "Earlier grades; include or exclude depending on the prediction scenario."],
        ["Target leakage warning", "Important", "Never include G3_math, G3_portuguese, or G3_target in X. Drop Subject for separate subject models."],
    ]
    for col in features:
        if "__" in col:
            original, category = col.split("__", 1)
            descriptions.append([col, "One-hot encoded", f"Indicator for category '{category}' of '{original}'."])
        elif col not in ("G3_math", "G3_portuguese", "G3_target"):
            descriptions.append([col, "Numeric", f"Numeric source feature: {col}."])

    OUTPUT.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(OUTPUT, engine="openpyxl") as writer:
        combined.to_excel(writer, sheet_name="Combined_Data", index=False)
        math.to_excel(writer, sheet_name="Mathematics", index=False)
        portuguese.to_excel(writer, sheet_name="Portuguese", index=False)
        pd.DataFrame(descriptions, columns=["Column", "Encoding / Type", "Meaning / Usage"]).to_excel(
            writer, sheet_name="Feature_Dictionary", index=False
        )
        for ws in writer.book.worksheets:
            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions
            for cell in ws[1]:
                cell.fill = PatternFill("solid", fgColor="1F4E78")
                cell.font = Font(color="FFFFFF", bold=True)
                cell.alignment = Alignment(wrap_text=True, vertical="center")
            ws.row_dimensions[1].height = 30
            for cells in ws.columns:
                width = max((len(str(c.value)) if c.value is not None else 0) for c in list(cells[:250]))
                ws.column_dimensions[get_column_letter(cells[0].column)].width = min(max(width + 2, 12), 36)
        ws = writer.book["Feature_Dictionary"]
        ws.column_dimensions["A"].width = 30
        ws.column_dimensions["B"].width = 22
        ws.column_dimensions["C"].width = 85

    print(f"Created {OUTPUT}")
    print(f"Mathematics rows: {len(math)}")
    print(f"Portuguese rows: {len(portuguese)}")
    print(f"Combined rows: {len(combined)}")
    print(f"Encoded feature columns: {len(features)}")

if __name__ == "__main__":
    main()


Project root: /Users/jaskirat_singh_grewal/Documents/IDC409_Project
Math CSV exists: True
Portuguese CSV exists: True
Output path: /Users/jaskirat_singh_grewal/Documents/IDC409_Project/data/student_data.xlsx
Created /Users/jaskirat_singh_grewal/Documents/IDC409_Project/data/student_data.xlsx
Mathematics rows: 395
Portuguese rows: 649
Combined rows: 1044
Encoded feature columns: 58
